# **Part 1 : Python & Data Foundations: ingest, validate, clean, enrich**

## Task 1.1

In [ ]:
from logging import exception
# Using only the csv module (not pandas): open appointments.csv with with open(..., newline=""),
# read it with csv.DictReader, and print the header and row count.
# Write a function parse_row(row) that converts age, fee_inr and prior_no_shows to int and distance_km to float,
# returning the converted dictionary. Loop over every row;
# use try/except ValueError to put rows that fail into a rejected list and the rest into an accepted list.
# Save rejected with json.dump(..., indent=2) as rejected_rows.json
import csv
import json

rejected = []
accepted = []

def parse_row(row):
      if all(row[key] != "" for key in row):
         row['age'] = int(row['age'])
         row['fee_inr'] = int(row['fee_inr'])
         row['prior_no_shows'] = int(row['prior_no_shows'])
         row['distance_km'] = float(row['distance_km'])

         accepted.append(row)

      else:
        rejected.append(row)

      return row


with open ("/content/followup_notes.csv","r",newline = "" ) as file:
    reader = csv.DictReader(file)

    print("Headers ",reader.fieldnames)

    for row in reader:
        try:
          converted_row = parse_row(row)
          accepted.append(converted_row)
        except ValueError:
          rejected.append(row)

print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))

with open("rejected_row.json",'w') as file:
    json.dump(rejected,file,indent = 2)


Headers  ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Accepted rows: 2004
Rejected rows: 36


##Task 1.2 — Inspect and clean, pass 1

In [ ]:
import pandas as pd

df = pd.read_csv("/content/followup_notes.csv")

df

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status
0,A0001,P509,C05,2025-01-24,2025-01-25,60.0,F,18.0,850,Phone,Consultation,Yes,1,No-Show
1,A0002,P046,C07,2025-05-27,2025-06-06,39.0,M,12.5,350,Walk-in,Follow-up,No,2,Attended
2,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
3,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
4,A0004,P482,C01,2025-04-18,2025-04-22,52.0,F,7.5,750,App,Consultation,Yes,0,Attended
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1015,A0996,P415,C05,2025-01-09,2025-01-31,20.0,M,13.0,700,Phone,Consultation,No,1,Attended
1016,A0997,P247,C05,2025-02-23,2025-02-27,77.0,M,17.4,400,App,Follow-up,Yes,3,No-Show
1017,A0998,P422,C06,2025-05-05,2025-05-16,58.0,M,3.2,850,Walk-in,Consultation,No,1,Attended
1018,A0999,P509,C05,2025-03-30,2025-04-12,59.0,F,180.0,300,Walk-in,Follow-up,Yes,0,Attended


In [ ]:
df.shape

(1020, 14)

In [ ]:
df.isnull().sum()

,0
appointment_id,0
patient_id,0
clinic_id,0
booked_on,0
appt_date,1
age,12
gender,0
distance_km,20
fee_inr,0
booking_channel,0


In [ ]:
df.duplicated().sum()

np.int64(20)

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()
print(df.shape)

(1000, 14)


In [ ]:
df['booking_channel'] = df['booking_channel'].str.strip().str.title()
print(df.shape)

(1000, 14)


/tmp/ipykernel_24874/1924164797.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['booking_channel'] = df['booking_channel'].str.strip().str.title()


In [ ]:
df['appt_date'] = pd.to_datetime(df['appt_date'],errors="coerce")
df['booked_on'] = pd.to_datetime(df['booked_on'])

print(df.shape)

(1000, 14)


/tmp/ipykernel_24874/863538835.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['appt_date'] = pd.to_datetime(df['appt_date'],errors="coerce")
/tmp/ipykernel_24874/863538835.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['booked_on'] = pd.to_datetime(df['booked_on'])


In [ ]:
df = df.dropna(subset=['appt_date'])
df = df.dropna(subset=['status'])
print(df.shape)

(990, 14)


##Task 1.3 — Clean, pass 2: validity, gaps and outliers

In [ ]:
import numpy as np

# (a) Treat invalid ages as missing
df.loc[(df['age'] < 1) | (df['age'] > 100), 'age'] = np.nan

# (b) Fill missing values with median
df['age'] = df['age'].fillna(df['age'].median()).round().astype(int)

df['distance_km'] = df['distance_km'].fillna(
    df['distance_km'].median()
)

# (c) Calculate quartiles, IQR and cap outliers

# Distance
Q1 = df['distance_km'].quantile(0.25)
Q3 = df['distance_km'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

distance_capped = (
    (df['distance_km'] < lower) |
    (df['distance_km'] > upper)
).sum()

df['distance_km'] = np.clip(df['distance_km'], lower, upper)

print("Distance values capped:", distance_capped)


# Fee
Q1 = df['fee_inr'].quantile(0.25)
Q3 = df['fee_inr'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

fee_capped = (
    (df['fee_inr'] < lower) |
    (df['fee_inr'] > upper)
).sum()

df['fee_inr'] = np.clip(df['fee_inr'], lower, upper)

print("Fee values capped:", fee_capped)


# (d) Round values
df['distance_km'] = df['distance_km'].round(1)

df['fee_inr'] = df['fee_inr'].round().astype(int)

print(df.head())


Distance values capped: 26
Fee values capped: 3
  appointment_id patient_id clinic_id  booked_on  appt_date  age gender  \
0          A0001       P509       C05 2025-01-24 2025-01-25   60      F   
1          A0002       P046       C07 2025-05-27 2025-06-06   39      M   
2          A0003       P009       C02 2025-03-24 2025-03-30   63      F   
4          A0004       P482       C01 2025-04-18 2025-04-22   52      F   
5          A0005       P251       C01 2024-11-23 2025-01-02   64      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In     Follow-up            No   
2          4.1      850         Walk-In  Consultation            No   
4          7.5      750             App  Consultation           Yes   
5         19.0      700           Phone  Consultation            No   

   prior_no_shows    status  
0               1   No-Show  
1             

##Task 1.4 — Features and a NumPy summary